### The LangChain conceptual guide on Memory
 details how AI agents remember past interactions, learn from user feedback, and adapt across conversations. In modern LangChain/LangGraph architecture, memory is split into Scope (Short-term vs. Long-term), Human-Analogy Types (Semantic, Episodic, Procedural), and Execution Strategies (In the hot path vs. In the background).

#### 1. Short-Term vs. Long-Term Memory
Short-Term Memory (Thread-Scoped):

Tracks an ongoing conversation within a single session/thread.

Managed as part of the agent's State in LangGraph and saved across execution steps using checkpointers.

Key Challenge: Long context windows increase latency, degrade model focus, and increase costs. Managing or filtering context window size is crucial.

Long-Term Memory (Cross-Thread / Application-Scoped):

Retains data across multiple sessions and user interactions using custom namespaces in a Store (e.g., organized by user_id or application_context).

Enables personalized experiences by sharing facts and rules across threads.

In [ ]:
from langgraph.graph import StateGraph, START, MessagesState
from langgraph.checkpoint.memory import MemorySaver
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

load_dotenv()  # Load environment variables from .env file

# 1. Define graph with MessagesState (handles message appending automatically)
builder = StateGraph(MessagesState)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7, max_tokens=500)


def call_model(state: MessagesState):
    # state["messages"] contains thread conversation history
    response = llm.invoke(state["messages"])
    return {"messages": [response]}


builder.add_node("agent", call_model)
builder.add_edge(START, "agent")

# 2. Compile with a checkpointer for short-term persistence
checkpointer = MemorySaver()
app = builder.compile(checkpointer=checkpointer)

# 3. Resume conversation by referencing thread_id
config = {"configurable": {"thread_id": "session_123"}}
app.invoke({"messages": [("user", "Hi, my name is Alice")]}, config)
app.invoke({"messages": [("user", "What is my name?")]}, config)

#### 2. Types of Long-Term Memory
LangChain categorizes long-term memory into three types based on cognitive psychology:

Semantic Memory (Facts & Concepts):

Stores facts about the user, domain, or environment to ground future responses.

Profile Approach: Maintained as a single, updating JSON object representing the user. Lower storage search complexity, but updating large profiles can be error-prone.

Collection Approach: Maintained as a list of distinct memory documents. Easier for models to generate and search (via vector embeddings/semantic search), but requires handling deduplication, updates, or deletions.

Episodic Memory (Experiences & Actions):

Remembers sequences of past actions or outcomes.

Implemented via few-shot prompt selection, pulling past input-output examples or agent execution logs into context to guide current tasks.

Procedural Memory (Instructions & Rules):

Defines how an agent operates (system prompts, agent code, model weights).

In practice, agents adapt procedurally via Reflection/Meta-prompting—updating their system prompt over time based on feedback.

##### A. Semantic Memory: Profile Approach

In [ ]:
from langgraph.store.base import BaseStore
from pydantic import BaseModel, Field


class ProfileSchema(BaseModel):
    name: str = Field(description="User's name")
    preferences: list[str] = Field(description="List of user preferences")


def update_user_profile(state: MessagesState, store: BaseStore, config: dict):
    user_id = config["configurable"]["user_id"]
    namespace = (user_id, "profile")

    # Fetch existing profile
    existing = store.get(namespace, "user_profile")
    current_profile = existing.value if existing else {"name": "", "preferences": []}

    # Prompt model to generate an updated JSON schema merging old data & new chat
    prompt = f"Current profile: {current_profile}\nChat history: {state['messages']}"
    updated_profile = llm.with_structured_output(ProfileSchema).invoke(prompt)

    # Overwrite single profile key
    store.put(namespace, "user_profile", updated_profile.dict())

##### B. Semantic Memory: Collection Approach

In [ ]:
import uuid
from langgraph.store.base import BaseStore


def add_memory_snippet(text: str, user_id: str, store: BaseStore):
    namespace = (user_id, "memories")
    memory_id = str(uuid.uuid4())

    # Store individual document snippet
    store.put(namespace, memory_id, {"content": text})


def search_user_memories(query: str, user_id: str, store: BaseStore):
    namespace = (user_id, "memories")

    # Semantic/vector search over user's collection
    results = store.search(namespace, query=query, limit=3)
    return [item.value["content"] for item in results]

##### C. Episodic Memory (Few-Shot Example Retrieval)

In [ ]:
def retrieve_few_shot_examples(user_input: str, store: BaseStore):
    # Namespace for task execution examples
    namespace = ("episodic_examples", "sql_queries")

    # Search for similar historical tasks
    matches = store.search(namespace, query=user_input, limit=2)

    few_shots = []
    for match in matches:
        few_shots.append(f"User: {match.value['input']}\nAssistant: {match.value['output']}")

    return "\n---\n".join(few_shots)

##### D. Procedural Memory (Reflection / Meta-Prompting)

In [ ]:
from langgraph.store.base import BaseStore


def run_agent_with_custom_prompt(state: MessagesState, store: BaseStore):
    namespace = ("agent_config",)

    # Retrieve current dynamic system prompt
    item = store.get(namespace, key="system_prompt")
    system_prompt = item.value["text"] if item else "Default helpful persona"

    messages = [("system", system_prompt)] + state["messages"]
    response = llm.invoke(messages)
    return {"messages": [response]}


def update_procedural_instructions(state: MessagesState, store: BaseStore):
    namespace = ("agent_config",)
    item = store.get(namespace, key="system_prompt")
    current_prompt = item.value["text"] if item else "Default helpful persona"

    # Reflect on conversation feedback and output a refined prompt
    reflection_prompt = f"Current Prompt: {current_prompt}\nFeedback/History: {state['messages']}"
    new_prompt = llm.invoke(reflection_prompt).content

    # Persist updated instructions
    store.put(namespace, "system_prompt", {"text": new_prompt})

#### 3. Execution Strategies for Writing Memories
In the Hot Path (Synchronous/Runtime):

Memory creation happens during conversation (e.g., using tool calls like save_memory).

Pros: Real-time availability and user visibility.

Cons: Increases response latency and forces the LLM to multitask between replying and managing memory.

In the Background (Asynchronous):

Memory creation runs asynchronously outside the main conversation loop (e.g., periodic cron jobs or batch reflection workers).

Pros: Zero added user latency and cleaner application logic.

Cons: Delay in memory availability across parallel threads.

##### A. In the Hot Path (Tool Calling)

In [ ]:
from langchain_core.tools import tool


@tool
def save_user_preference(preference: str, config: dict, store: BaseStore):
    """Call this tool to save explicit preferences stated by the user."""
    user_id = config["configurable"]["user_id"]
    namespace = (user_id, "preferences")
    memory_id = str(uuid.uuid4())

    store.put(namespace, memory_id, {"preference": preference})
    return f"Saved preference: {preference}"


# Bind tool to model
llm_with_tools = llm.bind_tools([save_user_preference])

##### B. In the Background (Asynchronous Batch Processing)

In [ ]:
async def extract_and_save_memories_async(thread_id: str, messages: list, store: BaseStore):
    """Runs asynchronously in the background (e.g., triggered via worker or queue)."""
    # 1. Extract useful facts offline
    extraction_prompt = f"Extract key facts from this conversation: {messages}"
    extracted_facts = await llm.ainvoke(extraction_prompt)

    # 2. Save facts to store without blocking main response thread
    namespace = (thread_id, "background_memories")
    await store.aput(namespace, "session_summary", {"summary": extracted_facts.content})


# Inside application server (e.g., FastAPI) after sending user response:
# asyncio.create_task(extract_and_save_memories_async(thread_id, state["messages"], store))